In [0]:
from pyspark.sql import functions as F

SILVER_PATH = "abfss://silver@ecommercedatalakebp01.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@ecommercedatalakebp01.dfs.core.windows.net"

In [0]:
df_returns = spark.read.format("delta").load(f"{SILVER_PATH}/returns")

print("total returns:", df_returns.count())
print("distinct order_id:", df_returns.select("order_id").distinct().count())

In [0]:
df_customers=spark.read.format("delta").load(f"{SILVER_PATH}/customers")
print(df_customers.columns)

In [0]:
df_customers=df_customers.select("customer_id","customer_name","email","city","state","country","signup_date")

In [0]:
special_rows=[("UNKNOWN","Unknown customer","Unknown","Unknown","Unknown","India",None),
              ("GUEST","Guest visitor","Unknown","Unknown","Unknown","India",None)]
df_special= spark.createDataFrame(special_rows,schema=df_customers.schema)
df_special.display()

In [0]:
dim_customers=df_customers.unionByName(df_special)

In [0]:
print("rows:", dim_customers.count())

In [0]:
assert dim_customers.select("customer_id").distinct().count() == dim_customers.count(),"Duplicates have been found"

In [0]:
dim_customers=dim_customers.withColumn("_processed_at", F.current_timestamp())

In [0]:
dim_customers.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/dim_customers")

In [0]:
check=spark.read.format("delta").load(f"{GOLD_PATH}/dim_customers")
print("rows:",check.count())
print("columns:", check.columns)

In [0]:
df_products = spark.read.format("delta").load(f"{SILVER_PATH}/products")
print(df_products.columns)

In [0]:
df_events=spark.read.format("delta").load(f"{SILVER_PATH}/website_events")


In [0]:
print(df_events.join(df_products,df_events["product_id"]==df_products["product_id"], "left_anti").count())

In [0]:
dim_products = (
    df_products
    .select("product_id", "product_name", "category", "price", "supplier_id")
    .withColumn("_processed_at", F.current_timestamp())
)
print(dim_products.columns)

In [0]:
assert dim_products.select("product_id").distinct().count() == dim_products.count(),"Duplicates have been found"


In [0]:
dim_products.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(f"{GOLD_PATH}/dim_products")

In [0]:
check = spark.read.format("delta").load(f"{GOLD_PATH}/dim_products")
print("rows:", check.count())
print("columns:", check.columns)

In [0]:
dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(F.to_date(F.lit("2023-01-01")), F.to_date(F.lit("2026-12-31")))
        ).alias("date")
    )
)

print("days:", dim_date.count())
dim_date.display()

In [0]:
dim_date = (
    dim_date
    .withColumn("year",        F.year("date"))
    .withColumn("quarter",     F.quarter("date"))
    .withColumn("month",       F.month("date"))
    .withColumn("month_name",  F.date_format("date", "MMMM"))
    .withColumn("year_month",  F.date_format("date", "yyyy-MM"))
    .withColumn("day",         F.dayofmonth("date"))
    .withColumn("day_name",    F.date_format("date", "EEEE"))
    .withColumn("is_weekend",  F.dayofweek("date").isin(1, 7))
)

dim_date.display()

In [0]:
dim_date = dim_date.withColumn("_processed_at", F.current_timestamp())
assert dim_date.select("date").distinct().count() == dim_date.count(), "Duplicate dates found!"
dim_date.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/dim_date")
check = spark.read.format("delta").load(f"{GOLD_PATH}/dim_date")
print("rows:", check.count())       
print("columns:", check.columns)     

In [0]:
df_returns=df_returns.select("order_id","return_date","return_reason","refund_amount")
df_orders=spark.read.format("delta").load(f"{SILVER_PATH}/orders")
df_orders=df_orders.select("order_id","customer_id","product_id","order_date","quantity","price","revenue","order_status")
df_payments=spark.read.format("delta").load(f"{SILVER_PATH}/payments")
df_payments=df_payments.select("order_id","payment_method","payment_status","payment_amount")
print("orders:", df_orders.count())       # 50000
print("payments:", df_payments.count())   # 50000
print("returns:", df_returns.count())     # 5910

In [0]:
fact_orders=(df_orders
             .join(df_payments, on="order_id",how="left")
             .join(df_returns, on="order_id",how="left"))
print("fact rows", fact_orders.count())


In [0]:
fact_orders.select("order_id", "payment_status", "return_reason", "refund_amount").display()

In [0]:
fact_orders = fact_orders.withColumn("is_returned", F.col("return_date").isNotNull())

print("returned orders:", fact_orders.filter(F.col("is_returned")).count())   # 5910

In [0]:
dim_customers = spark.read.format("delta").load(f"{GOLD_PATH}/dim_customers")
dim_products  = spark.read.format("delta").load(f"{GOLD_PATH}/dim_products")
dim_date      = spark.read.format("delta").load(f"{GOLD_PATH}/dim_date")
print(fact_orders.join(dim_customers, on="customer_id", how="left_anti").count())
print(fact_orders.join(dim_products, on="product_id", how="left_anti").count())
print(fact_orders.join(dim_date, fact_orders["order_date"]==dim_date["date"],"left_anti").count())

In [0]:
fact_orders=fact_orders.withColumn("_processed_at", F.current_timestamp())
assert fact_orders.select("order_id").distinct().count() == fact_orders.count(), "Duplicate orders found!"
fact_orders.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/fact_orders")
check = spark.read.format("delta").load(f"{GOLD_PATH}/fact_orders")
print("rows:", check.count())       
print("columns:", check.columns)

In [0]:
df_events = spark.read.format("delta").load(f"{SILVER_PATH}/website_events")    
fact_events=(df_events
             .filter(F.col("event_type")!="unknown_event")
             .select("event_id","customer_id","product_id","event_type","event_timestamp")
             .withColumn("event_date", F.to_date(F.col("event_timestamp")))
             .withColumn("_processed_at", F.current_timestamp()))
print("Fact rows:", fact_events.count())

In [0]:
print("customer orphans:", fact_events.join(dim_customers, on="customer_id", how="left_anti").count())
print("product orphans:",  fact_events.join(dim_products,  on="product_id",  how="left_anti").count())
print("date orphans:",     fact_events.join(dim_date, fact_events["event_date"] == dim_date["date"], "left_anti").count())

In [0]:
assert fact_events.select("event_id").distinct().count() == fact_events.count(), "Duplicate events found!"

fact_events.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/fact_events")

check = spark.read.format("delta").load(f"{GOLD_PATH}/fact_events")
print("rows:", check.count())
print("columns:", check.columns)